In [ ]:
import jax
import master.device as device
from master.obj import *
from master.estimator_bounds import *
import numpy as np
from master import *
from pty_defs import *

jax.config.update("jax_enable_x64", True)
cdtype = jnp.array(1j).dtype
jax.devices()
device_id = 1
device.setup_device(device_id=device_id)

In [ ]:
base_path = "/data/multispot_pty/pty-64-jitter-full/"
# base_path = "/data/multispot_pty/pty-64-jitter-10k/"
loader = Loader(base_path)

fig_path = "../docs/thesis/figures/pty"
base_fig_path = "../docs/thesis/figures"
SAVE_FIGS = True

In [ ]:
from master.digest import make_array_digest
from master.probes import construct_probe_fields, construct_probe_inf_probe_approx
from master.spotgen import create_multispot_array_product_jittered

pixel_intensity = 1e3
meas_id = f"CDI-intensity-{pixel_intensity:.3g}"
est_id = "ORACLE-second-order-ml"
est_label = "ORACLE+ML"


x_true, d, probes, spectral_support, _, measurement_params, measurement_hash_meas = (
    loader.get_measurement_with_probes(meas_id)
)

P_spectral_support = lambda x: ifft2_shifted(fft2_shifted(x) * spectral_support)

meas_id = f"CDI-intensity-{pixel_intensity:.3g}"
x_est, simulation_params, lls, x_est_raw, config_hash, measurement_hash_est = (
    loader.get_estimate_with_x_est_raw(meas_id, est_id)
)

# Check that measurement corresponds to reconstruction
assert measurement_hash_est == measurement_hash_meas
sim_params = simulation_params["simulation"]
probe_dict = simulation_params["probe"]
solver_params = simulation_params["solver"]
ml_params = solver_params["ml_stage"]

N_crop = sim_params["N_crop"]
N_pad = sim_params["N_pad"]
N_photons = sim_params["N_photons"]
N_samples = sim_params["N_samples"]
M = sim_params["M"]

NA_det = sim_params["NA_det"]

N_iter_ML = ml_params["N_iter_ML"]

probe_params = ProbeParams.from_dict(probe_dict)
print(probe_params)
pty_sim = PtySim(N_crop, N_pad, NA_det)
x, y, X, Y, R, k_0, k_x, k_y, K_X, K_Y, k, r_extent, k_extent = pty_sim.create_axes()
A, A_dagger, F_pupil, pupil_support = pty_sim.get_ops_and_pupil()

print(r_extent)
print(k_extent)

N_d_crop = d.shape[-1]

r_extent_cropped = list(
    create_physical_axes(
        center_crop(x_true, N_crop),
        lambda_0=pty_sim.lambda_0,
        pixel_size=pty_sim.lambda_0 / (2 * pty_sim.unit_l),
    )
)[-2]

# k_extent_pupil = list(
#     create_physical_axes(
#         ,
#         lambda_0=pty_sim.lambda_0,
#         pixel_size=pty_sim.lambda_0 / (2 * pty_sim.unit_l),
#     )
# )[-1]
# print(k_extent_pupil)

print(
    "Unit intensity probes and object digest: ",
    make_array_digest(probes),
    make_array_digest(x_true),
)

plot_intensity_phase_lambda(probes[0], r_extent)
base_probes, N_photons, N_coeff = pty_sim.create_base_probe(
    probe_params,
    pixel_intensity,
    f_probe=create_multispot_array_product_jittered,
    use_shift=True,
    r_frac_jitter=0.1,
)
base_probe = base_probes[0]

R_probes = pty_sim.get_R_probes(M, 0.5, probe_params)

x_true.shape, d.shape, x_est.shape, simulation_params["simulation"], N_d_crop

In [ ]:
plot_intensity_phase_spect_lambda(fft2_shifted(x_est[0]), k_extent)

# Ground truth object

In [ ]:
x_cameraman = cameraman() * 0.9 + 0.1
x_mandrill = as_phase_field(mandrill() ** 2 - 0.5)
x_obj, obj_support = crop_and_pad(x_cameraman * x_mandrill, N_crop=N_crop, N_pad=N_pad)
x_new_true = x_obj
x_obj_est_rec = P_spectral_support(x_true)
assert jnp.allclose(x_new_true, x_true, rtol=1e-12, atol=1e-14)
# assert jnp.allclose(x_obj_est_rec, x_obj_est, rtol=1e-12, atol=1e-14)
x_obj_est = x_obj_est_rec
if SAVE_FIGS:
    save_path = f"{fig_path}/pty_cameraman_object.png"
    save_est_path = f"{fig_path}/pty_cameraman_object_est.png"
    save_spect_path = f"{fig_path}/pty_cameraman_object_spect.png"
    save_est_spect_path = f"{fig_path}/pty_cameraman_object_est_spect.png"
    save_H_path = f"{fig_path}/pty_H.png"
else:
    save_path = None
    save_est_path = None
    save_spect_path = None
    save_est_spect_path = None
    save_H_path = None
x_obj_transfer = x_obj / jnp.max(jnp.abs(x_obj))
plot_intensity_phase_lambda(x_obj_transfer, save_path=save_path, extent=r_extent)
plot_intensity_phase_lambda(
    x_obj_est_rec, save_path=save_est_path, extent=r_extent, label=r"\tilde{x}"
)
plot_intensity_phase_spect_lambda(
    fft2_shifted(x_obj_transfer),
    save_path=save_spect_path,
    extent=k_extent,
    label=r"Fx",
)
plot_intensity_phase_spect_lambda(
    fft2_shifted(x_obj_est_rec) * spectral_support,
    save_path=save_est_spect_path,
    extent=k_extent,
    label=r"F\tilde{x}",
)
plot_lambda_spect(F_pupil, label=r"$zH$", extent=k_extent, save_path=save_H_path)

In [ ]:
base_probe_norm = base_probe / np.sqrt(M)
plot_intensity_phase_lambda(base_probe_norm, extent=r_extent)
spectrum_filtered_probe, spectral_support, P_spectral_support = (
    pty_sim.construct_spectral_support_rel(base_probe_norm)
)
plot_lambda_spect(
    np.abs(spectrum_filtered_probe),
    label=r"$|FP|*H$ ($\sqrt{\mathrm{photons}}$)",
    extent=k_extent,
    use_log=True,
    save_path=f"{fig_path}/pty_spectral_support_no_threshold_{pixel_intensity:.3g}.png",
)
plot_lambda_spect(
    spectral_support,
    label=r"$S'$",
    extent=k_extent,
    use_log=False,
    save_path=f"{fig_path}/pty_spectral_support_threshold_{pixel_intensity:.3g}.png",
)

# Single reconstruction and raw data

In [ ]:
from master.operators import make_crop_pad

N_pupil_crop = pupil_crop(F_pupil)
crop, pad = make_crop_pad(N_full=N_crop + 2 * N_pad, N_crop=N_pupil_crop)

pupil_shape = np.array(A(x_true).shape)
obj_shape = np.array(x_true.shape)
print(pupil_shape)
print(obj_shape)
shape_ratio = (pupil_shape / obj_shape)[0]
print(k_extent)
print(pupil_shape / obj_shape)
k_extent_pupil = [shape_ratio * k_lim for k_lim in k_extent]
k_extent_pupil

In [ ]:
plot_lambda(
    np.real(base_probe_norm),
    r_extent,
    label="P ($\sqrt{\mathrm{photons}}$)",
    save_path=f"{fig_path}/pty_probe_field_{pixel_intensity:.3g}.png",
)

In [ ]:
plot_lambda(
    np.abs(base_probe_norm) ** 2,
    r_extent,
    label="$|P|^2$ (photons)",
    save_path=f"{fig_path}/pty_probe_int_{pixel_intensity:.3g}.png",
)
plot_lambda_spect(
    np.abs(fft2_shifted(base_probe_norm)),
    label="$|FP|$ ($\sqrt{\mathrm{photons}}$)",
    extent=k_extent,
    save_path=f"{fig_path}/pty_probe_spect_field_{pixel_intensity:.3g}.png",
    use_log=False,
)

In [ ]:
plot_intensity_phase_lambda(
    base_probe_norm,
    r_extent,
    label="P",
    intensity_unit="photons",
    save_path=f"{fig_path}/pty_probe_{pixel_intensity:.3g}.png",
)

In [ ]:
from matplotlib.patches import Rectangle


def make_rect_patch(extent):
    l, r, b, t = extent
    return Rectangle((l, b), r - l, t - b, fill=False, ec="r", ls="--", lw=1.0)


intensity_sum = np.sum(np.abs(probes) ** 2, axis=0)
plot_lambda(
    intensity_sum,
    r_extent,
    label=r"$\sum_m |P^m|^2$ (photons)",
    patch=make_rect_patch(r_extent_cropped),
    save_path=f"{fig_path}/pty_probe_sum_{pixel_intensity:.3g}.png",
)
cropped_sum = center_crop(intensity_sum, N_crop)
plot_lambda(
    center_crop(intensity_sum, N_crop),
    r_extent_cropped,
    label=r"$\sum_m |P^m|^2$ (photons)",
)
print(np.min(cropped_sum), np.max(cropped_sum))

intensity_sum_obj = np.sum(np.abs(probes * x_true) ** 2, axis=0)
plot_lambda(
    intensity_sum_obj,
    r_extent,
    label=r"$\sum_m |z^m|^2$ (photons)",
    patch=make_rect_patch(r_extent_cropped),
    save_path=f"{fig_path}/pty_z_int_sum_{pixel_intensity:.3g}.png",
)
print(
    np.min(center_crop(intensity_sum_obj, N_crop)),
    np.max(center_crop(intensity_sum_obj, N_crop)),
)

intensity_sum_obj_est = np.sum(np.abs(probes * x_obj_est_rec) ** 2, axis=0)
plot_lambda(
    intensity_sum_obj_est,
    r_extent,
    label=r"$\sum_m |z^m|^2$ (photons)",
    patch=make_rect_patch(r_extent_cropped),
    save_path=f"{fig_path}/pty_z_from_xest_int_sum_{pixel_intensity:.3g}.png",
)
print(
    np.min(center_crop(intensity_sum_obj_est, N_crop)),
    np.max(center_crop(intensity_sum_obj_est, N_crop)),
)

In [ ]:
print(d.shape, x_est.shape)
if SAVE_FIGS:
    z_path = f"{fig_path}/pty_object_z_{pixel_intensity:.3g}.png"
    y_path = f"{fig_path}/pty_object_y_{pixel_intensity:.3g}.png"
    d_path = f"{fig_path}/pty_sample_d_{pixel_intensity:.3g}.png"
else:
    z_path, y_path, d_path = None, None, None
plot_intensity_phase_lambda(
    probes[0] * x_true,
    label="z^m",
    save_path=y_path,
    intensity_unit="expected counts",
    extent=k_extent,
    use_two_pi_extent=True,
)

plot_intensity_phase_lambda(
    pad(A(probes[0] * x_true)),
    use_log=True,
    label="y^m",
    save_path=y_path,
    intensity_unit="expected counts",
    x_label=FREQ_X_LAMBDA_LABEL,
    y_label=FREQ_Y_LAMBDA_LABEL,
    extent=k_extent,
    use_two_pi_extent=True,
)
plot(
    pad(d[0, 0, :]),
    use_log=True,
    label="d (counts)",
    save_path=d_path,
    x_label=FREQ_X_LAMBDA_LABEL,
    y_label=FREQ_Y_LAMBDA_LABEL,
    extent=k_extent,
    use_two_pi_extent=True,
)

# $\ell$ plots

In [ ]:
from master.stats import mean_with_standard_error
from master.mle import (
    log_likelihood,
    log_likelihood_pty,
    log_likelihood_x,
    calc_ell_init,
)
from scipy.stats import gmean

PLOT_SECOND_ORDER = False
USE_GMEAN = False
fig_ell_path = "../docs/thesis/figures/"


def plot_ll_convergence_fo(
    pixel_intensity=1e3,
    init="ORACLE",
    y_c_bottom=0.75,
    y_c_top=1.25,
    legend_loc="lower right",
):
    meas_id = f"CDI-intensity-{pixel_intensity:.3g}"
    (
        x_true,
        d,
        probes,
        spectral_support,
        x_obj_est,
        measurement_params,
        measurement_hash_meas,
    ) = loader.get_measurement_with_probes(meas_id)
    ell_d = log_likelihood_pty(d, np.sqrt(d))

    def norm_ll(lls):
        return (ell_d - lls) / ell_d

    N_alphas = 16
    test = False
    if test:
        alphas = [0.1]
    else:
        alphas = np.logspace(-4, 1, N_alphas)
    init_est_id = f"{init}"
    if init == "ORACLE":
        init_est = np.broadcast_to(
            x_obj,
            (N_samples,) + x_obj.shape,
        )
    else:
        init_est, _, _, _, _ = loader.get_estimate(meas_id, init_est_id)

    print(init_est.shape, probes.shape, d.shape)

    # Full reduction does not fit in VRAM
    # TODO: develop exact CPU mirror of forward model and use this instead
    ell_init = calc_ell_init(d, init_est, N_samples, A, probes)

    norm_ell_init = norm_ll(ell_init)
    ll_upper_bound, _ = mean_with_standard_error(norm_ell_init, USE_GMEAN, -1)
    print(ell_init.shape)
    print(ll_upper_bound)
    min_ll = ll_upper_bound
    fig = plt.figure(figsize=(FIG_W, FIG_H), layout="compressed")
    if PLOT_SECOND_ORDER:
        est_id = f"{init}-second-order-ml"
        # est_label = "ORACLE+FS"
        x_est, simulation_params, lls, config_hash, measurement_hash_est = (
            loader.get_estimate(meas_id, est_id)
        )
        # Idea: trivial maximum of dataset is y = sqrt(d).
        # Normalize using this to get somewhat invariant measure of convergence
        normed_ll = norm_ll(lls)
        mean_norm_ll, se_norm_ll = mean_with_standard_error(normed_ll, USE_GMEAN, -1)
        # For gmean, check if correct

        print(lls)
        min_ll = np.minimum(np.nanmin(mean_norm_ll), min_ll)
        print(mean_norm_ll, se_norm_ll, mean_norm_ll / se_norm_ll)

        plt.errorbar(
            range(1, lls.shape[0] + 1),
            mean_norm_ll,
            yerr=se_norm_ll,
            label=rf"$\alpha = 1.0$",
        )
    else:
        for alpha in alphas:
            if alpha > 2.5:
                continue
            est_id = f"{init}-first-order-ml-{alpha:.3g}"
            # est_label = "ORACLE+FS"
            x_est, simulation_params, lls, config_hash, measurement_hash_est = (
                loader.get_estimate(meas_id, est_id)
            )
            # Idea: trivial maximum of dataset is y = sqrt(d).
            # Normalize using this to get somewhat invariant measure of convergence
            normed_ll = norm_ll(lls)
            mean_norm_ll, se_norm_ll = mean_with_standard_error(
                normed_ll, USE_GMEAN, -1
            )
            min_ll = np.minimum(np.nanmin(mean_norm_ll), min_ll)

            plt.errorbar(
                range(1, lls.shape[0] + 1),
                mean_norm_ll,
                yerr=se_norm_ll,
                label=rf"$\alpha = {alpha:.3g}$",
            )

    plt.xlabel(r"$i$")
    plt.ylabel(r"$\tilde{\ell}(\hat{x}^{(i)})$")
    plt.xscale("log")
    plt.yscale("log")
    print(min_ll, ll_upper_bound)
    plt.ylim(min_ll * y_c_bottom, ll_upper_bound * y_c_top)
    plt.xlim(left=1)
    plt.legend(loc=legend_loc)
    if SAVE_FIGS:
        suffix = "_gmean" if USE_GMEAN else ""
        save_path = (
            f"{fig_ell_path}/pty-ell/ell_{init}{suffix}_{pixel_intensity:.3g}.png"
        )
    else:
        save_path = None

    if save_path is not None:
        fig.savefig(save_path, dpi=PLOT_DPI)
    plt.show()

In [ ]:
plot_ll_convergence_fo(1e3, "rPIE", 0.975, 1.05)

In [ ]:
plot_ll_convergence_fo(1e10, "rPIE", 0.985, 1.2, "lower left")

In [ ]:
plot_ll_convergence_fo(1e3, "ORACLE", 0.985, 1.01, "lower left")
plot_ll_convergence_fo(1e10, "ORACLE", 0.985, 1.01, "lower left")

# Estimator evaluation

In [ ]:
est_base_path = f"{base_fig_path}/pty/estimator"


def evaluate_estimator_and_save_plots(
    x_model_ml, x_obj, obj_support, label="HIO+ML", extent=None
):
    make_path = lambda x: f"{est_base_path}/{label}/{x}_{pixel_intensity:.3g}.png"
    n_hat = np.abs(x_model_ml) ** 2 - jnp.var(
        x_model_ml, axis=0, ddof=1, keepdims=True
    )  # / N_samples
    n = jnp.abs(x_obj) ** 2
    n_hat_mu, n_hat_std, n_hat_bias, n_hat_MSE = estimator_stats(n_hat, n)
    print("n")
    plot_lambda(
        n_hat_mu,
        label=r"$\operatorname{Mean}(\hat{n})$",
        extent=extent,
        save_path=make_path("nhat_mean"),
    )
    # plot_lambda(n, label=r"$n$", extent=extent)
    plot_lambda(
        n_hat_bias,
        label=r"$\operatorname{Bias}(\hat{n})$",
        extent=extent,
        save_path=make_path("nhat_bias"),
    )
    plot_lambda(
        n_hat_std,
        label=r"$\operatorname{Std}(\hat{n})$",
        extent=extent,
        save_path=make_path("nhat_std"),
    )
    plot_lambda(
        n_hat_MSE,
        label=r"$\operatorname{MSE}(\hat{n})$",
        extent=extent,
        save_path=make_path("nhat_MSE"),
    )

    print("phi")

    phi_hat = np.angle(x_model_ml) * obj_support
    phi = jnp.angle(x_obj) * obj_support

    phase_about_mean = True

    # Circular mean of the phase estimates
    phi_hat_mu = np.angle(np.mean(np.exp(1j * phi_hat), axis=0))
    wrap = lambda a: np.angle(np.exp(1j * a))

    if phase_about_mean:
        # Deviations about the circular mean; bias = circular mean vs truth
        phi_hat_dev = wrap(phi_hat - phi_hat_mu)
        phi_hat_bias = wrap(phi_hat_mu - phi)
        phi_hat_std = np.std(phi_hat_dev, axis=0, ddof=1)
        phi_hat_MSE = phi_hat_bias**2 + phi_hat_std**2
    else:
        # Errors wrapped about the true phase
        phi_hat_error = wrap(phi_hat - phi)
        phi_hat_bias = np.mean(phi_hat_error, axis=0)
        phi_hat_std = np.std(phi_hat_error, axis=0, ddof=1)
        phi_hat_MSE = np.mean(phi_hat_error**2, axis=0)

    # # Circular mean of the phase estimates
    # phi_hat_mu = np.angle(np.mean(np.exp(1j * phi_hat), axis=0))

    # # Wrapped phase errors in (-pi, pi]
    # phi_hat_error = np.angle(np.exp(1j * (phi_hat - phi)))

    # # Statistics in the local phase coordinate
    # phi_hat_bias = np.mean(phi_hat_error, axis=0)
    # phi_hat_std = np.std(phi_hat_error, axis=0, ddof=1)
    # phi_hat_MSE = np.mean(phi_hat_error**2, axis=0)

    plot_lambda(
        phi_hat_mu,
        label=r"$\operatorname{Mean}(\hat{\phi})$",
        extent=extent,
        save_path=make_path("phihat_mean"),
    )
    plot_lambda(
        phi_hat_bias,
        label=r"$\operatorname{Bias}(\hat{\phi})$",
        extent=extent,
        save_path=make_path("phihat_bias"),
    )
    plot_lambda(
        phi_hat_std,
        label=r"$\operatorname{Std}(\hat{\phi})$",
        extent=extent,
        save_path=make_path("phihat_std"),
    )
    plot_lambda(
        phi_hat_MSE,
        label=r"$\operatorname{MSE}(\hat{\phi})$",
        extent=extent,
        save_path=make_path("phihat_MSE"),
    )
    return n_hat_mu, n_hat_std, n_hat_bias, phi_hat_mu, phi_hat_std, phi_hat_bias


pixel_intensity = 1e3
meas_id = f"CDI-intensity-{pixel_intensity:.3g}"

for est_label, est_id in [
    ("ORACLE+ML", "ORACLE-second-order-ml"),
    ("rPIE+ML", "rPIE-second-order-ml"),
]:
    x_est, simulation_params, lls, x_est_raw, config_hash, measurement_hash_est = (
        loader.get_estimate_with_x_est_raw(meas_id, est_id=est_id)
    )

    n_hat_mu, n_hat_std, n_hat_bias, phi_hat_mu, phi_hat_std, phi_hat_bias = (
        evaluate_estimator_and_save_plots(
            center_crop(P_spectral_support(x_est_raw), N_crop),
            center_crop(x_obj_est, N_crop),
            center_crop(obj_support, N_crop),
            label=est_label,
            extent=r_extent_cropped,
        )
    )

In [ ]:
# Verify that all plots are the same resolution
d = r"~/master/docs/thesis/figures/pty/estimator/rPIE+ML"
!file {d}/*.png

In [ ]:
from master.operators import make_crop_pad


def calc_pty_bounds_x_true_est(x_true_est, probes):
    N_pupil_crop = pupil_crop(F_pupil)
    crop, pad = make_crop_pad(N_full=N_crop + 2 * N_pad, N_crop=N_pupil_crop)
    cropped_pupil = crop(F_pupil)
    ys_true = A(probes * x_true_est)
    S, Q = ptycho_SQ(probes, ys_true, pad, cropped_pupil)
    Jnn, Jphiphi, Jnphi = fisher_pty_2x2_polar(S, Q, x_true_est)
    print(probes.shape, ys_true.shape, cropped_pupil.shape)
    Jpinv_nn, Jpinv_phiphi, Jpinv_nphi = ls_fs_cov(
        probes, ys_true, pad, x_true_est, cropped_pupil
    )[1]
    Jn_inv_2x2, Jphi_inv_2x2 = diag_of_2x2_inverse(Jnn, Jnphi, Jnphi, Jphiphi)

    Jnn_rpa, Jphiphi_rpa, _ = fisher_pty_random_phase(
        probes, ys_true.shape[-2:], x_true_est, cropped_pupil
    )[1]
    return (
        Jn_inv_2x2,
        Jphi_inv_2x2,
        Jpinv_nn,
        Jpinv_phiphi,
        1.0 / Jnn_rpa,
        1.0 / Jphiphi_rpa,
    )


Jn_inv_2x2, Jphi_inv_2x2, Jpinv_nn, Jpinv_phiphi, Jn_rpa_inv, Jphi_rpa_inv = (
    calc_pty_bounds_x_true_est(x_obj_est, probes)
)

In [ ]:
n_std_2x2_crb = jnp.sqrt(Jn_inv_2x2) * obj_support
phi_std_2x2_crb = jnp.sqrt(Jphi_inv_2x2) * obj_support

label_submatrix = r"\text{pixel}"
label_pinv = r"\text{EWCRB}"
label_rpa = r"\text{RPA}"
plot_lambda(
    center_crop(n_std_2x2_crb, N_crop),
    label=r"$\sigma^{" + label_submatrix + r"}_{\hat{n}}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_n_2x2_inv_{pixel_intensity:.3g}.png",
)
plot_lambda(
    center_crop(phi_std_2x2_crb, N_crop),
    label=r"$\sigma^{" + label_submatrix + r"}_{\hat{\phi}}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_phi_2x2_inv_{pixel_intensity:.3g}.png",
)

In [ ]:
n_std_crb = jnp.sqrt(Jpinv_nn) * obj_support
phi_std_crb = jnp.sqrt(Jpinv_phiphi) * obj_support

plot_lambda(
    center_crop(n_std_crb, N_crop),
    label=r"$\sigma^{" + label_pinv + r"}_{\hat{n}}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_n_pinv_CRB_{pixel_intensity:.3g}.png",
)
plot_lambda(
    center_crop(phi_std_crb, N_crop),
    label=r"$\sigma^{" + label_pinv + r"}_{\hat{\phi}}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_phi_pinv_CRB_{pixel_intensity:.3g}.png",
)

In [ ]:
n_std_crb = jnp.sqrt(Jn_rpa_inv) * obj_support
phi_std_crb = jnp.sqrt(Jphi_rpa_inv) * obj_support
plot_lambda(
    center_crop(n_std_crb, N_crop),
    label=r"$\sigma^{" + label_rpa + r"}_{\hat{n}}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_n_random_phase_{pixel_intensity:.3g}.png",
)
plot_lambda(
    center_crop(phi_std_crb, N_crop),
    label=r"$\sigma^{" + label_rpa + r"}_{\hat{\phi}}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_phi_random_phase_{pixel_intensity:.3g}.png",
)

In [ ]:
def ratio_plot(
    theta_hat_std, theta_std_bound, theta_label=r"n", bound_label=label_submatrix
):
    plot_lambda(
        theta_hat_std / center_crop(theta_std_bound, N_crop),
        label=r"$\operatorname{Std}[\hat{"
        + theta_label
        + r"}] / \sigma_{\hat{"
        + theta_label
        + r"}}^{"
        + bound_label
        + r"}$",
        extent=r_extent_cropped,
    )


ratio_plot(n_hat_std, n_std_2x2_crb, r"n", label_submatrix)
ratio_plot(phi_hat_std, phi_std_2x2_crb, r"\phi", label_submatrix)

In [ ]:
ratio_plot(n_hat_std, n_std_crb, r"n", label_pinv)
ratio_plot(phi_hat_std, phi_std_crb, r"\phi", label_pinv)

# Repeat in $u = \Re{x}, v = \Im{x}$ parametrization

In [ ]:
est_base_path = f"{base_fig_path}/pty/estimator"


def evaluate_bias_spectrum(
    x_model_ml, x_obj, obj_support, factor, label="HIO+ML", extent=None
):
    make_path = lambda x: f"{est_base_path}/{label}/{x}_{pixel_intensity:.3g}.png"
    u_hat = np.real(x_model_ml)
    v_hat = np.imag(x_model_ml)
    u = np.real(x_obj)
    v = np.imag(x_obj)
    # Re x / u block.

    u_hat_mu, u_hat_std, u_hat_bias, u_hat_MSE = estimator_stats(u_hat, u)

    # plot_lambda_spect(jnp.abs() * spectral_support,         label=r"$|F(\operatorname{Avg}(\hat{u}))|$",use_log=True,extent=k_extent,save_path=make_path("uhat_mean_spect"))

    # Im x / v block.
    v_hat_mu, v_hat_std, v_hat_bias, v_hat_MSE = estimator_stats(v_hat, v)
    print("DEBUG PLOT")
    u_bias_spect = fft2_shifted(u_hat_bias)
    plot_lambda_spect(
        jnp.abs(u_bias_spect) * spectral_support,
        label=r"$|F(\operatorname{Bias}(\hat{u}))|$",
        use_log=True,
        extent=k_extent,
        save_path=make_path("uhat_bias_spect"),
    )
    v_bias_spect = fft2_shifted(v_hat_bias)
    plot_lambda_spect(
        jnp.abs(v_bias_spect) * spectral_support,
        label=r"$|F(\operatorname{Bias}(\hat{v}))|$",
        use_log=True,
        extent=k_extent,
        save_path=make_path("vhat_bias_spect"),
    )


pixel_intensity = 1e10
meas_id = f"CDI-intensity-{pixel_intensity:.3g}"

for est_label, est_id in [
    ("ORACLE+ML", "ORACLE-second-order-ml"),
    ("rPIE+ML", "rPIE-second-order-ml"),
]:
    x_est, simulation_params, lls, x_est_raw, config_hash, measurement_hash_est = (
        loader.get_estimate_with_x_est_raw(meas_id, est_id=est_id)
    )
    evaluate_bias_spectrum(
        x_est,
        x_obj_est,
        obj_support,
        1.0,
        label=est_label,
        extent=r_extent_cropped,
    )

In [ ]:
est_base_path = f"{base_fig_path}/pty/estimator"


def evaluate_estimator_Re_Im_and_save_plots(
    x_model_ml, x_obj, obj_support, factor, label="HIO+ML", extent=None
):
    make_path = lambda x: f"{est_base_path}/{label}/{x}_{pixel_intensity:.3g}.png"
    u_hat = np.real(x_model_ml)
    v_hat = np.imag(x_model_ml)
    u = np.real(x_obj)
    v = np.imag(x_obj)
    # Re x / u block.

    u_hat_mu, u_hat_std, u_hat_bias, u_hat_MSE = estimator_stats(u_hat, u)

    # plot_lambda_spect(jnp.abs() * spectral_support,         label=r"$|F(\operatorname{Avg}(\hat{u}))|$",use_log=True,extent=k_extent,save_path=make_path("uhat_mean_spect"))

    # Im x / v block.
    v_hat_mu, v_hat_std, v_hat_bias, v_hat_MSE = estimator_stats(v_hat, v)
    print("DEBUG PLOT")
    u_bias_spect = fft2_shifted(pad_image(u_hat_bias, N_pad))
    plot_lambda_spect(
        jnp.abs(u_bias_spect) * spectral_support,
        label=r"$|F(\operatorname{Bias}(\hat{u}))|$",
        use_log=True,
        extent=k_extent,
        save_path=make_path("uhat_bias_spect"),
    )
    v_bias_spect = fft2_shifted(pad_image(v_hat_bias, N_pad))
    plot_lambda_spect(
        jnp.abs(v_bias_spect) * spectral_support,
        label=r"$|F(\operatorname{Bias}(\hat{v}))|$",
        use_log=True,
        extent=k_extent,
        save_path=make_path("vhat_bias_spect"),
    )
    # std_spect = crop_and_pad(fft2_shifted(u_hat_std), N_crop, N_pad)[0]
    # plot_lambda_spect(
    #     jnp.abs(std_spect) ** 2 * spectral_support * spectral_support, use_log=False,extent=k_extent,
    # )
    # bias_spect = crop_and_pad(fft2_shifted(v_hat_bias), N_crop, N_pad)[0]
    # plot_intensity_phase(bias_spect * spectral_support, use_log=False)
    print("u")
    plot_lambda(
        u_hat_mu * factor,
        label=r"$\operatorname{Mean}(\hat{u})$",
        extent=extent,
        save_path=make_path("uhat_mean"),
    )
    plot_lambda(
        u_hat_bias * factor,
        label=r"$\operatorname{Bias}(\hat{u})$",
        extent=extent,
        save_path=make_path("uhat_bias"),
    )
    plot_lambda(
        u_hat_std * factor,
        label=r"$\operatorname{Std}(\hat{u})$",
        extent=extent,
        save_path=make_path("uhat_std"),
    )
    plot_lambda(
        u_hat_MSE * factor**2,
        label=r"$\operatorname{MSE}(\hat{u})$",
        extent=extent,
        save_path=make_path("uhat_MSE"),
    )

    print("v")
    plot_lambda(
        v_hat_mu * factor,
        label=r"$\operatorname{Mean}(\hat{v})$",
        extent=extent,
        save_path=make_path("vhat_mean"),
    )
    plot_lambda(
        v_hat_bias * factor,
        label=r"$\operatorname{Bias}(\hat{v})$",
        extent=extent,
        save_path=make_path("vhat_bias"),
    )
    plot_lambda(
        v_hat_std * factor,
        label=r"$\operatorname{Std}(\hat{v})$",
        extent=extent,
        save_path=make_path("vhat_std"),
    )
    plot_lambda(
        v_hat_MSE * factor**2,
        label=r"$\operatorname{MSE}(\hat{v})$",
        extent=extent,
        save_path=make_path("vhat_MSE"),
    )

    return (
        u_hat_mu,
        u_hat_std,
        u_hat_bias,
        u_hat_MSE,
        v_hat_mu,
        v_hat_std,
        v_hat_bias,
        v_hat_MSE,
    )


pixel_intensity = 1e10
meas_id = f"CDI-intensity-{pixel_intensity:.3g}"

for est_label, est_id in [
    ("ORACLE+ML", "ORACLE-second-order-ml"),
    ("rPIE+ML", "rPIE-second-order-ml"),
]:
    x_est, simulation_params, lls, x_est_raw, config_hash, measurement_hash_est = (
        loader.get_estimate_with_x_est_raw(meas_id, est_id=est_id)
    )

    (
        u_hat_mu,
        u_hat_std,
        u_hat_bias,
        u_hat_MSE,
        v_hat_mu,
        v_hat_std,
        v_hat_bias,
        v_hat_MSE,
    ) = evaluate_estimator_Re_Im_and_save_plots(
        center_crop(x_est, N_crop),
        center_crop(x_obj_est, N_crop),
        center_crop(obj_support, N_crop),
        1.0,
        label=est_label,
        extent=r_extent_cropped,
    )

# Compare CRB from $J_x^+$ with lower bound from $1/\operatorname{diag}(J_x)$

In [ ]:
from master.operators import make_crop_pad


def calc_pty_MSE_bounds_x_true_est_uv(x_true_est, probes):
    N_pupil_crop = pupil_crop(F_pupil)
    crop, pad = make_crop_pad(N_full=N_crop + 2 * N_pad, N_crop=N_pupil_crop)
    cropped_pupil = crop(F_pupil)
    ys_true = A(probes * x_true_est)
    S, Q = ptycho_SQ(probes, ys_true, pad, cropped_pupil)
    Juu, Jvv, Juv = fisher_pty_2x2_cart(S, Q)
    print(probes.shape, ys_true.shape, cropped_pupil.shape)
    Jpinv_uu, Jpinv_vv, Jpinv_uv = ls_fs_cov(
        probes, ys_true, pad, x_true_est, cropped_pupil
    )[0]
    Ju_inv_2x2, Jv_inv_2x2 = diag_of_2x2_inverse(Juu, Juv, Juv, Jvv)

    Juu_rpa, Jvv_rpa, _ = fisher_pty_random_phase(
        probes, ys_true.shape[-2:], x_true_est, cropped_pupil
    )[0]
    return Ju_inv_2x2, Jv_inv_2x2, Jpinv_uu, Jpinv_vv, 1.0 / Juu_rpa, 1.0 / Jvv_rpa


Ju_inv_2x2, Jv_inv_2x2, Jpinv_uu, Jpinv_vv, Ju_rpa_inv, Jv_rpa_inv = (
    calc_pty_MSE_bounds_x_true_est_uv(x_obj_est, probes)
)

In [ ]:
x_u_std_crb = jnp.sqrt(Jpinv_uu) * obj_support
x_v_std_crb = jnp.sqrt(Jpinv_vv) * obj_support

plot_lambda(
    center_crop(x_u_std_crb, N_crop),
    label=r"$\sigma_{\hat{u}}^{" + label_pinv + r"}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_u_pinv_CRB_{pixel_intensity:.3g}.png",
)
plot_lambda(
    center_crop(x_v_std_crb, N_crop),
    label=r"$\sigma_{\hat{v}}^{" + label_pinv + r"}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_v_pinv_CRB_{pixel_intensity:.3g}.png",
)
# Visualize to show that per-pixel information is perfectly uniform in this parametrization
# plot_lambda(
#     det_j * obj_support,
#     label=r"$[det(J_{jj})]$",
# )

In [ ]:
x_u_std_diag = jnp.sqrt(Ju_inv_2x2) * obj_support
x_v_std_diag = jnp.sqrt(Jv_inv_2x2) * obj_support
plot_lambda(
    center_crop(x_u_std_diag, N_crop),
    label=r"$\sigma_{\hat{u}}^{" + label_submatrix + r"}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_u_2x2_inv_{pixel_intensity:.3g}.png",
)
plot_lambda(
    center_crop(x_v_std_diag, N_crop),
    label=r"$\sigma_{\hat{v}}^{" + label_submatrix + r"}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_v_2x2_inv_{pixel_intensity:.3g}.png",
)

In [ ]:
x_u_std_diag = jnp.sqrt(Ju_rpa_inv) * obj_support
x_v_std_diag = jnp.sqrt(Jv_rpa_inv) * obj_support
plot_lambda(
    center_crop(x_u_std_diag, N_crop),
    label=r"$\sigma_{\hat{u}}^{" + label_rpa + r"}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_u_random_phase_{pixel_intensity:.3g}.png",
)
plot_lambda(
    center_crop(x_v_std_diag, N_crop),
    label=r"$\sigma_{\hat{v}}^{" + label_rpa + r"}$",
    extent=r_extent_cropped,
    save_path=f"{base_fig_path}/bounds/pty_v_random_phase_{pixel_intensity:.3g}.png",
)

In [ ]:
ratio_plot(u_hat_std, x_u_std_diag, r"u", label_submatrix)
ratio_plot(v_hat_std, x_v_std_diag, r"v", label_submatrix)

In [ ]:
ratio_plot(u_hat_std, x_u_std_crb, r"u", label_pinv)
ratio_plot(v_hat_std, x_v_std_crb, r"v", label_pinv)

In [ ]:
def bound_ratio_plot(
    theta_std_bound_1,
    theta_std_bound_2,
    theta_label="n",
    bound_label_1=label_pinv,
    bound_label_2=label_submatrix,
):
    plot(
        center_crop(theta_std_bound_1, N_crop) / center_crop(theta_std_bound_2, N_crop),
        label=r"$\sigma_{\hat{"
        + theta_label
        + r"}}^{"
        + bound_label_1
        + r"} / \sigma_{\hat{"
        + theta_label
        + r"}{"
        + bound_label_2
        + r"}$",
        extent=r_extent_cropped,
    )


bound_ratio_plot(x_u_std_diag, x_u_std_crb, r"u")
bound_ratio_plot(x_v_std_diag, x_v_std_crb, r"v")